In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 93.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import DataLoader, TensorDataset
from torch.func import functional_call

import pennylane as qml
import torch.optim as optim


In [ ]:
import random

torch.manual_seed(90)


1- Preprocessing

label encoding

In [ ]:
# label encoding
df = pd.read_csv('DATASET-balanced.csv')
le = LabelEncoder()
df['LABEL'] = le.fit_transform(df['LABEL'])
display(df.head())

,chroma_stft,rms,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,mfcc1,mfcc2,mfcc3,mfcc4,...,mfcc12,mfcc13,mfcc14,mfcc15,mfcc16,mfcc17,mfcc18,mfcc19,mfcc20,LABEL
0,0.338055,0.027948,2842.948867,4322.916759,6570.586186,0.041050,-462.169586,90.311272,19.073769,24.046888,...,-6.686564,0.902086,-7.251551,-1.198342,4.747403,-4.986279,0.953935,-5.013138,-6.779060,0
1,0.443766,0.037838,2336.129597,3445.777044,3764.949874,0.047730,-409.413422,120.348808,-7.161531,5.114784,...,-2.131157,-6.876417,-1.359395,0.326401,-5.420016,-2.109968,-1.757634,-9.537907,-8.494421,0
2,0.302528,0.056578,2692.988386,2861.133180,4716.610271,0.080342,-318.996033,120.490273,-24.625771,23.891073,...,-5.853725,-3.724773,-6.627182,-5.117002,-6.072106,-0.994653,-1.617120,-3.922354,-7.033001,0
3,0.319933,0.031504,2241.665382,3503.766175,3798.641521,0.047180,-404.636749,136.320908,2.308172,-3.907071,...,-1.898315,-2.046493,-7.176277,-3.293508,4.209121,0.121835,-5.407063,-3.654926,-3.274857,0
4,0.420055,0.016158,2526.069123,3102.659519,5025.077899,0.051905,-410.497925,152.731400,-18.266771,51.993462,...,-1.952340,0.810868,6.238493,6.555839,7.535542,2.849219,2.616843,-1.793357,-5.060998,0


In [ ]:
# stratfied sampling
X = df.drop('LABEL', axis = 1)
y = df['LABEL']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42) # saving 80% of the dataset for training

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = 42) # 10% of the dataset for validation and 10% for testing

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# checking the balance of classes
print("\nClass distribution in y_train:")
print(pd.Series(y_train).value_counts())

print("\nClass distribution in y_val:")
print(pd.Series(y_val).value_counts())

print("\nClass distribution in y_test:")
print(pd.Series(y_test).value_counts())

In [ ]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)



In [ ]:
train_features = torch.tensor(X_train, dtype=torch.float32)
train_labels = torch.tensor(y_train.values, dtype=torch.long)

val_features = torch.tensor(X_val, dtype=torch.float32)
val_labels = torch.tensor(y_val.values, dtype=torch.long)

test_features = torch.tensor(X_test, dtype=torch.float32)
test_labels = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)



BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


2 - Model

In [ ]:

n_qubits = 12
n_blocks = 12

indices = torch.arange(2**n_qubits)
bitstrings = ((indices.unsqueeze(1) >> torch.arange(n_qubits-1, -1, -1)) & 1).to(torch.float64)

dev = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev, interface="torch", diff_method="backprop")
def pqc(phi):
    for b in range(n_blocks):
        for i in range(n_qubits):
            qml.RY(phi[b, i], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

    return qml.probs(wires=range(n_qubits))


def generate_pqc_outputs(phi):
    probs = pqc(phi)
    pqc_results = torch.cat([bitstrings, probs.unsqueeze(1)], dim=1)

    return pqc_results


In [ ]:
class MappingModel(nn.Module):
    def __init__(self):
        super(MappingModel, self).__init__()

        self.layer1 = nn.Linear(13, 20)
        self.leaky_relu = nn.LeakyReLU(0.01)

        self.layer2 = nn.Linear(20, 1)

    def forward(self, x):
        """
        x: Tensor de formato (4096, 13)
        Retorna: Pesos brutos theta_i para a CNN
        """
        x = self.leaky_relu(self.layer1(x))
        return self.layer2(x)

In [ ]:
class ScalingModel(nn.Module):
    def __init__(self, num_scales=8):
        super(ScalingModel, self).__init__()
        self.scales = nn.Parameter(torch.full((num_scales,), 1.0, dtype=torch.float64))

    def forward(self, trimmed_weights, layer_split_points):

        weight_chunks = torch.split(trimmed_weights, layer_split_points)

        final_weights = []

        for i, chunk in enumerate(weight_chunks):
            scaled_block = chunk * self.scales[i]
            final_weights.append(scaled_block)

        return torch.cat(final_weights)




In [ ]:
class ClassicalCNN(nn.Module):
    def __init__(self, num_features=26, num_classes=2):
        super(ClassicalCNN, self).__init__()



        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(kernel_size=2) # 26 -> 13


        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2) # 13 -> 6 (floor)


        self.flat_features = 32 * 6


        self.fc1 = nn.Linear(self.flat_features, 10)
        self.fc2 = nn.Linear(10, num_classes)

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.pool1(f.leaky_relu(self.conv1(x)))
        x = self.pool2(f.leaky_relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = f.leaky_relu(self.fc1(x))
        x = self.fc2(x)
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


cnn_model = ClassicalCNN()
print(count_parameters(cnn_model))

layer_split_points = [48, 16, 1536, 32, 1920, 10, 20, 2]



In [ ]:

mapping_model = MappingModel().to(torch.float64)

for m in mapping_model.modules():
    if isinstance(m, nn.Linear):
        torch.nn.init.normal_(m.weight, mean=0.0, std=0.5) # tava em 0.5
        torch.nn.init.constant_(m.bias, 0.0) #tava em 0.0






phi_trainable = torch.empty((n_blocks, n_qubits), dtype=torch.float64, requires_grad=True)

with torch.no_grad():
    torch.nn.init.uniform_(phi_trainable, -0.01, 0.01)




scaling_model = ScalingModel(num_scales=len(layer_split_points)).to(torch.float64)
cnn_model = ClassicalCNN()

optimizer = torch.optim.Adam(
    list(mapping_model.parameters()) +
    list(scaling_model.parameters()) +
    [phi_trainable],
    lr=0.001
)


criterion = nn.CrossEntropyLoss()

num_epochs = 100

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    correct_train = 0
    total_train = 0
    mapping_model.train()
    scaling_model.train()

    for batch_idx, (batch_audio, labels) in enumerate(train_loader):
        optimizer.zero_grad()


        pqc_input = generate_pqc_outputs(phi_trainable)

        brute_weights = mapping_model(pqc_input).squeeze(-1)

        trimmed_weights = brute_weights[:3584]


        final_weights_f64 = scaling_model(trimmed_weights, layer_split_points)
        final_weights_f64.retain_grad()


        final_weights_f32 = final_weights_f64.to(torch.float32)

        chunks = torch.split(final_weights_f32, layer_split_points)
        params_dict = {
            name: chunk.view(p.shape)
            for (name, p), chunk in zip(cnn_model.named_parameters(), chunks)
        }

        predictions = functional_call(cnn_model, params_dict, (batch_audio,))

        loss = criterion(predictions, labels)

        loss.backward()

        if batch_idx == 0:
            print(f"  Batch {batch_idx+1} Gradients:")
            if phi_trainable.grad is not None:
                print(f"    phi_trainable mean abs grad: {phi_trainable.grad.abs().mean():.6f}")
            if mapping_model.layer1.weight.grad is not None:
                print(f"    MappingModel.layer1.weight mean abs grad: {mapping_model.layer1.weight.grad.abs().mean():.6f}")
            if mapping_model.layer2.weight.grad is not None:
                print(f"    MappingModel.layer2.weight mean abs grad: {mapping_model.layer2.weight.grad.abs().mean():.6f}")
            if scaling_model.scales.grad is not None:
                print(f"    ScalingModel.scales mean abs grad: {scaling_model.scales.grad.abs().mean():.6f}")
            if final_weights_f64.grad is not None:
                print(f"    final_weights_f64 mean abs grad (after scaling/before CNN): {final_weights_f64.grad.abs().mean():.6f}")

            _, predicted = torch.max(predictions.data, 1)
            print(f"  Batch {batch_idx+1} Labels (True vs Predicted):")
            print(f"    True:      {labels[:5].tolist()}")
            print(f"    Predicted: {predicted[:5].tolist()}")

        optimizer.step()

        epoch_loss += loss.item()
        _, predicted = torch.max(predictions.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_losses.append(epoch_loss / len(train_loader))

    mapping_model.eval()
    scaling_model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for batch_idx_val, (batch_audio_val, labels_val) in enumerate(val_loader):
            pqc_input_val = generate_pqc_outputs(phi_trainable)
            brute_weights_val = mapping_model(pqc_input_val).squeeze(-1)
            trimmed_weights_val = brute_weights_val[:3584]
            final_weights_f64_val = scaling_model(trimmed_weights_val, layer_split_points)
            final_weights_f32_val = final_weights_f64_val.to(torch.float32)

            chunks_val = torch.split(final_weights_f32_val, layer_split_points)
            params_dict_val = {
                name: chunk.view(p.shape)
                for (name, p), chunk in zip(cnn_model.named_parameters(), chunks_val)
            }
            predictions_val = functional_call(cnn_model, params_dict_val, (batch_audio_val,))
            loss_val = criterion(predictions_val, labels_val)
            val_loss += loss_val.item()

            _, predicted_val = torch.max(predictions_val.data, 1)
            total_val += labels_val.size(0)
            correct_val += (predicted_val == labels_val).sum().item()

            if batch_idx_val == 0:
                print(f"  Validation Batch {batch_idx_val+1} Labels (True vs Predicted):")
                print(f"    True:      {labels_val[:5].tolist()}")
                print(f"    Predicted: {predicted_val[:5].tolist()}")


    val_losses.append(val_loss / len(val_loader))
    val_accuracy = 100 * correct_val / total_val
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1} | Avg Train Loss: {train_losses[-1]:.4f} | Train Acc: {100 * correct_train / total_train:.2f}% | Avg Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_accuracy:.2f}%")


print("\n--- Training Complete --- ")
mapping_model.eval()
scaling_model.eval()
test_loss = 0.0
correct_test = 0
total_test = 0


with torch.no_grad():
    for batch_audio_test, labels_test in test_loader:
        pqc_input_test = generate_pqc_outputs(phi_trainable)
        brute_weights_test = mapping_model(pqc_input_test).squeeze(-1)
        trimmed_weights_test = brute_weights_test[:3584]
        final_weights_f64_test = scaling_model(trimmed_weights_test, layer_split_points)
        final_weights_f32_test = final_weights_f64_test.to(torch.float32)

        chunks_test = torch.split(final_weights_f32_test, layer_split_points)
        params_dict_test = {
            name: chunk.view(p.shape)
            for (name, p), chunk in zip(cnn_model.named_parameters(), chunks_test)
        }
        predictions_test = functional_call(cnn_model, params_dict_test, (batch_audio_test,))
        loss_test = criterion(predictions_test, labels_test)
        test_loss += loss_test.item()

        _, predicted_test = torch.max(predictions_test.data, 1)
        total_test += labels_test.size(0)
        correct_test += (predicted_test == labels_test).sum().item()

final_test_loss = test_loss / len(test_loader)
final_test_accuracy = 100 * correct_test / total_test

print(f"Final Test Loss: {final_test_loss:.4f} | Final Test Accuracy: {final_test_accuracy:.2f}%")